# Notebook 02 — Sequence Data Formats: FASTA, FASTQ, and the File Ecosystem

**Module:** 08 — Bioinformatics: Sequence Analysis  
**Notebook:** 02 of 18  
**Time estimate:** 60 minutes

---
## Step 1 — Motivation

Before you can align sequences, you need to read them. FASTA and FASTQ are the two
universal currencies of computational biology. Every NGS pipeline, BLAST search,
and alignment tool speaks these formats. Understanding them at the byte level — not
just "Biopython can parse them" — is a baseline Track A expectation.

---
## Step 2 — Intuition

FASTA: a header line starting with `>`, followed by the sequence, possibly wrapped
at 60 or 80 characters. That's the whole format.

FASTQ: the same idea, but each record has 4 lines — header, sequence, `+` separator,
and a quality string. The quality string encodes per-base confidence as ASCII characters.

The other formats you'll encounter: SAM/BAM (aligned reads), VCF (variants), BED
(genomic intervals), GFF/GTF (gene annotations). Each is a specialization of the same
idea: tab-separated metadata about genomic positions.

---
## Step 3 — Biological Background

Sequencing machines produce reads — short (Illumina: 50–300 bp) or long (PacBio/
Oxford Nanopore: 1–100 kb) strings of nucleotide calls. Each call has an associated
quality: how confident is the base-caller that this is an A, not a G?

The Phred quality score Q encodes this: Q = -10 log₁₀(P_error). A Q30 score means
1-in-1000 probability of a wrong call. Q20 means 1-in-100. Standard sequencing
quality thresholds: Q20 minimum, Q30 "good".

FASTQ encodes Q as `chr(Q + 33)` — ASCII offset 33 (the `!` character) maps to Q=0.

---
## Step 4 — Mathematical Explanation

**Phred quality score:**

$$Q = -10 \log_{10}(P_{\text{error}})$$

| Q | P(error) | Accuracy |
|---|----------|----------|
| 10 | 0.1 | 90% |
| 20 | 0.01 | 99% |
| 30 | 0.001 | 99.9% |
| 40 | 0.0001 | 99.99% |

**ASCII encoding:** FASTQ Sanger format (Illumina 1.8+) uses offset 33.
Character `I` = ASCII 73 → Q = 73 - 33 = 40.
Character `!` = ASCII 33 → Q = 0.

---
## Step 5 — Computational Explanation

Parsing FASTA: read lines; when you see `>`, start a new record; accumulate sequence
lines until next `>` or EOF.

Parsing FASTQ: read in groups of 4 lines. Line 1 = header (`@`), line 2 = sequence,
line 3 = `+`, line 4 = quality string. Validate that len(seq) == len(qual).

Both formats can be gzipped (`.fa.gz`, `.fastq.gz`). Use `gzip.open()` with mode `'rt'`.

In [ ]:
# Step 6 — Python Implementation
from __future__ import annotations
import gzip
from dataclasses import dataclass
from typing import Iterator
import math


@dataclass
class FastaRecord:
    header: str
    sequence: str

    def __len__(self) -> int:
        return len(self.sequence)

    def gc_content(self) -> float:
        seq = self.sequence.upper()
        gc = seq.count('G') + seq.count('C')
        return gc / len(seq) if seq else 0.0


@dataclass
class FastqRecord:
    header: str
    sequence: str
    quality_string: str

    @property
    def phred_scores(self) -> list[int]:
        return [ord(c) - 33 for c in self.quality_string]

    @property
    def mean_quality(self) -> float:
        scores = self.phred_scores
        return sum(scores) / len(scores)

    def error_probabilities(self) -> list[float]:
        return [10 ** (-q / 10) for q in self.phred_scores]

    def trim_quality(self, min_q: int = 20) -> 'FastqRecord':
        scores = self.phred_scores
        # Trim from right
        end = len(scores)
        while end > 0 and scores[end - 1] < min_q:
            end -= 1
        # Trim from left
        start = 0
        while start < end and scores[start] < min_q:
            start += 1
        return FastqRecord(
            self.header,
            self.sequence[start:end],
            self.quality_string[start:end]
        )


def parse_fasta(filepath: str) -> Iterator[FastaRecord]:
    opener = gzip.open if filepath.endswith('.gz') else open
    current_header = None
    seq_parts = []
    with opener(filepath, 'rt') as fh:
        for line in fh:
            line = line.rstrip()
            if line.startswith('>'):
                if current_header is not None:
                    yield FastaRecord(current_header, ''.join(seq_parts))
                current_header = line[1:]
                seq_parts = []
            else:
                seq_parts.append(line.upper())
        if current_header is not None:
            yield FastaRecord(current_header, ''.join(seq_parts))


def parse_fastq(filepath: str) -> Iterator[FastqRecord]:
    opener = gzip.open if filepath.endswith('.gz') else open
    with opener(filepath, 'rt') as fh:
        while True:
            header = fh.readline().rstrip()
            if not header:
                break
            seq = fh.readline().rstrip().upper()
            _ = fh.readline()  # the '+' line
            qual = fh.readline().rstrip()
            if len(seq) != len(qual):
                raise ValueError(f"Length mismatch: seq={len(seq)}, qual={len(qual)}")
            yield FastqRecord(header.lstrip('@'), seq, qual)


def phred_to_prob(q: int) -> float:
    return 10 ** (-q / 10)


def prob_to_phred(p: float) -> int:
    return round(-10 * math.log10(p))


print("Phred Q30:", phred_to_prob(30), "(expect 0.001)")
print("Phred Q20:", phred_to_prob(20), "(expect 0.01)")
print("prob 0.001 -> Q:", prob_to_phred(0.001), "(expect 30)")

In [ ]:
# Test with synthetic FASTA and FASTQ data (written to strings, not files)

fasta_text = """>Human_BRCA1
ATGGATTTATCTGCTCTTCGCGTTGAAGAAGTACAAAATGTCATTAATGCTATGCAGAAAATCTTAGA
GTGTCCCATCTGTCTGGAGTTGATCAAGGAACCTGTCTCCACAAAGTGTGACCACATATTTTGCAAAT
>Mouse_Brca1
ATGGATTTATCTGCTCTTCGCGTTGAAGAAGTACAAAATGTCGTTAATGCTATGCAGAAAATCTTAGA
GTGTCCCATCTGTCTGGAGTTGATCAAGGAACCTGTCTCCACAAAGTGTGACCACATATTTTGCAAAT
"""

fastq_text = """@read_1 length=50
ATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCG
+
IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII
@read_2 length=50
GCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTA
+
!!!!!!!!!!!!!!!!!!!!IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII
"""

# Parse from string using StringIO
from io import StringIO


def parse_fasta_string(text: str) -> list[FastaRecord]:
    records = []
    current_header = None
    seq_parts = []
    for line in StringIO(text):
        line = line.rstrip()
        if line.startswith('>'):
            if current_header is not None:
                records.append(FastaRecord(current_header, ''.join(seq_parts)))
            current_header = line[1:]
            seq_parts = []
        elif line:
            seq_parts.append(line.upper())
    if current_header is not None:
        records.append(FastaRecord(current_header, ''.join(seq_parts)))
    return records


def parse_fastq_string(text: str) -> list[FastqRecord]:
    records = []
    lines = [l.rstrip() for l in StringIO(text) if l.rstrip()]
    for i in range(0, len(lines), 4):
        if i + 3 >= len(lines):
            break
        header = lines[i].lstrip('@')
        seq = lines[i + 1].upper()
        qual = lines[i + 3]
        records.append(FastqRecord(header, seq, qual))
    return records


fasta_records = parse_fasta_string(fasta_text)
for r in fasta_records:
    print(f"{r.header}: len={len(r)}, GC={r.gc_content():.3f}")

print()
fastq_records = parse_fastq_string(fastq_text)
for r in fastq_records:
    scores = r.phred_scores
    print(f"{r.header}: mean_Q={r.mean_quality:.1f}, min_Q={min(scores)}, max_Q={max(scores)}")
    trimmed = r.trim_quality(min_q=20)
    print(f"  After Q20 trim: {len(r.sequence)} -> {len(trimmed.sequence)} bp")

In [ ]:
# Step 7 — Visualization: quality score distribution from synthetic reads
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

rng = np.random.default_rng(42)

# Simulate 200 reads with realistic Illumina quality profiles
# Quality typically starts high, drops at 3' end
n_reads = 200
read_length = 75

positions = np.arange(read_length)
# Q score: starts at 38, decays to ~25 at 3' end with noise
base_quality = 38 - positions * 0.15

all_quals = []
for _ in range(n_reads):
    q = base_quality + rng.normal(0, 3, read_length)
    q = np.clip(q, 0, 40).astype(int)
    all_quals.append(q)

all_quals = np.array(all_quals)

fig = plt.figure(figsize=(12, 8))
gs = gridspec.GridSpec(2, 2, fig)

# Panel A: Quality per position (box plot style)
ax1 = fig.add_subplot(gs[0, 0])
ax1.fill_between(positions, np.percentile(all_quals, 25, axis=0),
                 np.percentile(all_quals, 75, axis=0), alpha=0.3, label='IQR')
ax1.plot(positions, np.median(all_quals, axis=0), 'b-', lw=2, label='Median')
ax1.axhline(30, color='g', ls='--', label='Q30')
ax1.axhline(20, color='orange', ls='--', label='Q20')
ax1.set_xlabel('Position in read')
ax1.set_ylabel('Phred quality score')
ax1.set_title('A. Per-position quality (synthetic Illumina)')
ax1.legend(fontsize=8)

# Panel B: Phred score -> P(error) transformation
ax2 = fig.add_subplot(gs[0, 1])
q_vals = np.arange(0, 41)
p_err = 10 ** (-q_vals / 10)
ax2.semilogy(q_vals, p_err, 'r-', lw=2)
ax2.axvline(20, color='orange', ls='--', label='Q20 (1% error)')
ax2.axvline(30, color='g', ls='--', label='Q30 (0.1% error)')
ax2.set_xlabel('Phred Q score')
ax2.set_ylabel('P(base call error)')
ax2.set_title('B. Phred score to error probability')
ax2.legend(fontsize=8)

# Panel C: Mean quality distribution across reads
ax3 = fig.add_subplot(gs[1, 0])
mean_quals = all_quals.mean(axis=1)
ax3.hist(mean_quals, bins=20, color='steelblue', edgecolor='white')
ax3.axvline(30, color='g', ls='--', label='Q30')
ax3.set_xlabel('Mean read quality')
ax3.set_ylabel('Count')
ax3.set_title('C. Mean quality distribution (200 reads)')
ax3.legend()

# Panel D: FASTQ ASCII encoding table
ax4 = fig.add_subplot(gs[1, 1])
example_chars = '!"#$%&\'()*+,-./0123456789:;<=>?@ABCDEFGHIJKLMNOPQRSTUVWXYZ'
q_from_ascii = [ord(c) - 33 for c in example_chars]
ax4.bar(range(len(q_from_ascii)), q_from_ascii, width=1,
        color=['g' if q >= 30 else 'orange' if q >= 20 else 'r' for q in q_from_ascii])
ax4.set_xlabel('Character index')
ax4.set_ylabel('Phred Q score')
ax4.set_title('D. Sanger FASTQ ASCII encoding (offset 33)\nGreen≥Q30, Orange≥Q20, Red<Q20')
ax4.axhline(30, color='g', ls='--', alpha=0.5)
ax4.axhline(20, color='orange', ls='--', alpha=0.5)

plt.tight_layout()
plt.savefig('sequence_data_formats.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved.")

---
## Step 8 — Exercises

See `exercises/02_formats_exercises.md`.

1. Write a function `filter_reads(records, min_length, min_mean_quality)` that returns
   only reads passing both thresholds. Test it on the synthetic data above.
2. What is the Phred encoding of the character `A`? Of `I`? Of `!`? (No Google —
   compute from `ord()`.)
3. How would you detect whether a FASTQ file uses Sanger (offset 33) vs. Solexa
   (offset 64) encoding? Write a heuristic.
4. Write a function that converts a FASTQ record to FASTA (drop the quality string).

---
## Step 10 — Quiz

1. What does the `+` line in FASTQ contain, and why does it exist?
2. A base has Q=25. What is its error probability? Its accuracy?
3. A FASTQ file has reads with quality string starting `!!`. Should you worry? Why?
4. What is the difference between a FASTA and a FASTQ record?
5. Why are FASTQ files typically gzipped but FASTA files less often so?

---
## Step 12 — Reflection

> *[In one paragraph: what is the Phred score formula, and why does the log scale make
> intuitive sense for representing sequencing quality?]*

---
## Step 13 — Papers Referenced

- Ewing & Green (1998) "Base-calling of automated sequencer traces using Phred."
  *Genome Research* — the original Phred paper.

---
*Next: `03_edit_distance_and_alignment_problem.ipynb`*